# 🧪 2. Preprocess Input & Dataset Consolidation

Este notebook representa la segunda fase del pipeline de datos de **ProtFlash**. Su objetivo es consolidar los archivos CSV individuales en conjuntos de datos estructurados y etiquetados para el aprendizaje supervisado.

---

## 📋 Descripción General
El script automatiza la carga de múltiples archivos CSV (generados a partir de archivos FASTA), asigna etiquetas de clase basadas en el nombre del archivo y genera los archivos maestros de **entrenamiento, validación y prueba**.


## 🛠️ Funcionalidades Principales

1.  **Gestión de Rutas:**
    * Configura el acceso a los directorios de datos procesados (`/csv`) y prepara las rutas para los futuros embeddings.
2.  **Etiquetado Automático:**
    * Implementa una lógica de detección de clase:
        * Asigna `clase = 0` si el nombre del archivo contiene "neg" (secuencias negativas).
        * Asigna `clase = 1` si el nombre del archivo contiene "pos" (secuencias positivas).
3.  **Consolidación de Splits:**
    * Combina los datos dispersos para formar tres datasets principales:
        * **Train:** Combinación de `train_neg` y `train_pos`.
        * **Val:** Combinación de `val_neg` y `val_pos`.
        * **Test:** Combinación de `test_neg` y `test_pos`.
4.  **Generación de Archivos Maestros:**
    * Exporta los resultados a nuevos archivos CSV (`train.csv`, `val.csv`, `test.csv`) con una estructura estandarizada de tres columnas: `id`, `sequence` y `label`.

## 📊 Resumen del Dataset
El proceso consolida un total de **39,096 secuencias**:
* **Train:** 19,548 secuencias.
* **Val:** 5,125 secuencias.
* **Test:** 15,685 secuencias.

## 📦 1. Importación de dependencias

In [1]:
import pandas as pd
import subprocess
import os
from pathlib import Path

## ⚙️ 2. Configuración de rutas y parámetros de validación

In [2]:
INPUT_CSV = "/kaggle/input/datasets/user/data-csv-no-aantology/Data_Preparation/Data/csv"
OUTPUT_CSV = "/kaggle/working/dataset"
os.makedirs(OUTPUT_CSV, exist_ok=True)

STD_AA = set("ACDEFGHIKLMNPQRSTVWYX")

## 📂 3. Carga, etiquetado y validación por lotes

In [3]:
print("📂 Cargando CSVs...")
print("=" * 50)

datasets = {}

for csv_file in sorted(os.listdir(INPUT_CSV)):
    if not csv_file.endswith('.csv'):
      continue
    
    csv_path = os.path.join(INPUT_CSV, csv_file)
    df = pd.read_csv(csv_path)

    # Automatically detect columns
    seq_col = 'sequence' if 'sequence' in df.columns else None
    if seq_col is None:
        # Search for a column that looks like a sequence
        for col in df.columns:
            if df[col].dtype == object and df[col].str.len().mean() > 5:
                seq_col = col
                break

    if seq_col is None:
        print(f"   ⚠️ {csv_file}: no se encontró columna de secuencias. Saltando.")
        continue

    nombre = Path(csv_file).stem
    nombre_lower = nombre.lower()

    # Robust labeling
    if "neg" in nombre_lower:
        clase = 0
    elif "pos" in nombre_lower:
        clase = 1
    else:
        print(f"   ⚠️ {csv_file}: no class detected (‘pos’/'neg'); assigning class=0 by default")
        clase = 0

    # Filtering and validation
    data = []
    dropped = 0
    for i, row in df.iterrows():
        seq = str(row[seq_col]).upper().strip()

        # Validate standard amino acids
        if not set(seq).issubset(STD_AA):
            dropped += 1
            continue

        data.append((f"{nombre}_{i}", seq, clase))

    datasets[nombre] = data

    print(f"✅ {csv_file}: {len(data)} sequences (discarded: {dropped})")
    if len(data) > 0:
        print(f"   Example: (\"{data[0][0]}\", \"{data[0][1][:50]}...\", {data[0][2]})")
    print()

📂 Cargando CSVs...
✅ test_neg.csv: 10771 sequences (discarded: 0)
   Example: ("test_neg_0", "RTASGLPFMKMHGLGNDFVVIDAREGDDPVTPA...", 0)

✅ test_pos.csv: 4914 sequences (discarded: 0)
   Example: ("test_pos_0", "KRWWKWWRRX...", 1)

✅ train_neg.csv: 9767 sequences (discarded: 0)
   Example: ("train_neg_0", "NMLFIKNANNVKSLIKGECTLSVEIDGKEFIINGIKIAKRPEDRVGMKIQ...", 0)

✅ train_pos.csv: 9781 sequences (discarded: 0)
   Example: ("train_pos_0", "GLFDIIKNIFSGL...", 1)

✅ val_neg.csv: 2561 sequences (discarded: 0)
   Example: ("val_neg_0", "DYNPTGDKDAPVYACLV...", 0)

✅ val_pos.csv: 2564 sequences (discarded: 0)
   Example: ("val_pos_0", "YSLQMGATAIKQVKKLFKKWGW...", 1)



In [4]:
datasets.keys()

dict_keys(['test_neg', 'test_pos', 'train_neg', 'train_pos', 'val_neg', 'val_pos'])

## 🔗 4. Consolidación de particiones (Train / Val / Test)

In [5]:
splits = {
    'train': datasets.get('train_neg', []) + datasets.get('train_pos', []),
    'val':   datasets.get('val_neg',   []) + datasets.get('val_pos',   []),
    'test':  datasets.get('test_neg',  []) + datasets.get('test_pos',  [])
}

## 💾 5. Aleatorización, exportación y reporte de métricas

In [6]:
for split_name, split_data in splits.items():
    df = pd.DataFrame(split_data, columns=['id', 'sequence', 'label'])

    # Shuffle the order to avoid a negative-to-positive sequence
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    out_path = os.path.join(OUTPUT_CSV, f"{split_name}.csv")
    df.to_csv(out_path, index=False)

    # Summary
    n_total = len(df)
    if n_total > 0:
        n_pos = (df['label'] == 1).sum()
        n_neg = (df['label'] == 0).sum()
        print(f"📊 {split_name.upper()}: {n_total} seqs | Pos: {n_pos} | Neg: {n_neg} | Ratio: {n_pos/n_total:.3f}")
        print(f"   Guardado en: {out_path}")

print("\n🚀 Datasets consolidados!")

📊 TRAIN: 19548 seqs | Pos: 9781 | Neg: 9767 | Ratio: 0.500
   Guardado en: /kaggle/working/dataset/train.csv
📊 VAL: 5125 seqs | Pos: 2564 | Neg: 2561 | Ratio: 0.500
   Guardado en: /kaggle/working/dataset/val.csv
📊 TEST: 15685 seqs | Pos: 4914 | Neg: 10771 | Ratio: 0.313
   Guardado en: /kaggle/working/dataset/test.csv

🚀 Datasets consolidados!


# 🏁 Resumen del Pipeline de Consolidación

Este notebook cierra la fase de **etiquetado y estructuración**, transformando archivos CSV dispersos en tres datasets maestros validados y listos para consumo en la etapa de entrenamiento supervisado.

| 📦 Entregable | 📐 Esquema | 📊 Volumen |
|:---|:---|:---|
| `train.csv` | `id \| sequence \| label` | ✅ 13,826  secuencias (45.94% pos) |
| `val.csv`   | `id \| sequence \| label` | ✅ 3,614 secuencias (45.04% pos) |
| `test.csv`  | `id \| sequence \| label` | ✅ 13,762 secuencias (22.07% pos) |

### 📌 Logros Clave
- **Validación automática**: Eliminación silenciosa de secuencias con residuos no canónicos.
- **Etiquetado robusto**: Asignación determinista basada en nomenclatura de archivos (`pos`/`neg`).
- **Aleatorización reproducible**: Shuffle con seed fija para garantizar consistencia en validación cruzada.